# 3-D Fluid Dynamics with Heat Transfer: Cartesian and Cylindrical Coordinates

## ChBE 3300: Multidimensional Fluids and Heat Transport

### Overview

This notebook covers two important cases of 3-D fluid dynamics with heat transfer relevant to chemical and biomolecular engineering:

1. **Steady-State Cylindrical**: Blood flow in an artery with metabolic heat transfer (Biomolecular)
2. **Transient Cartesian**: Natural convection in a heated bioreactor (Chemical Engineering)

### Learning Objectives
- Understand coupled momentum and energy equations in 3-D
- Implement finite difference methods for fluid flow and heat transfer
- Visualize velocity and temperature fields in 3-D
- Apply results to biomedical and chemical engineering design problems

### Governing Equations

For incompressible flow with heat transfer, we solve:
- **Continuity**: $\nabla \cdot \vec{u} = 0$
- **Momentum (Navier-Stokes)**: $\rho\left(\frac{\partial \vec{u}}{\partial t} + \vec{u} \cdot \nabla \vec{u}\right) = -\nabla p + \mu \nabla^2 \vec{u} + \rho \vec{g}$
- **Energy**: $\rho c_p\left(\frac{\partial T}{\partial t} + \vec{u} \cdot \nabla T\right) = k \nabla^2 T + q_{gen}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set style
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")
print("NumPy version:", np.__version__)

---
## Part 1: Steady-State Flow in Cylindrical Coordinates

### Application: Blood Flow in Artery with Heat Transfer (Biomolecular)

Consider blood flow through a cylindrical artery:
- Blood enters at body core temperature (37°C)
- Arterial wall is cooler due to proximity to skin surface (35°C)
- Metabolic activity in surrounding tissue generates heat
- Fully developed laminar flow (Poiseuille flow)

### Clinical Significance
Understanding heat transfer in blood vessels is important for:
- Hypothermia and hyperthermia treatment
- Drug delivery systems (temperature-sensitive drugs)
- Cryotherapy and thermal ablation procedures
- Perfusion analysis in medical diagnostics

### Governing Equations (Cylindrical Coordinates: r, θ, z)

For fully developed, axisymmetric flow (no variation with θ):

**Momentum (z-direction, steady):**
$$\frac{dp}{dz} = \mu \frac{1}{r}\frac{d}{dr}\left(r\frac{du_z}{dr}\right)$$

**Analytical velocity profile (Poiseuille flow):**
$$u_z(r) = \frac{1}{4\mu}\left(-\frac{dp}{dz}\right)(R^2 - r^2)$$

**Energy equation (with convection):**
$$\rho c_p u_z \frac{\partial T}{\partial z} = k\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial T}{\partial r}\right) + \frac{\partial^2 T}{\partial z^2}\right] + q_{met}$$

where $q_{met}$ is metabolic heat generation.

### Simplifications for this Model
1. Axial conduction often negligible (high Peclet number): $\frac{\partial^2 T}{\partial z^2} \approx 0$
2. Entrance region approximation (developing thermal boundary layer)
3. Constant properties

In [ ]:
# Problem parameters - Blood Flow in Artery
# Geometric parameters
R_artery = 0.003  # Artery radius (m) - 3 mm (medium-sized artery)
L_artery = 0.05   # Length considered (m) - 5 cm

# Blood properties (at body temperature)
rho_blood = 1060.0      # Density (kg/m³)
mu_blood = 0.0035       # Dynamic viscosity (Pa·s) - approximate for blood
k_blood = 0.52          # Thermal conductivity (W/m·K)
cp_blood = 3600.0       # Specific heat (J/kg·K)
alpha_blood = k_blood / (rho_blood * cp_blood)

# Flow parameters
dp_dz = -50.0           # Pressure gradient (Pa/m)
u_max = -dp_dz * R_artery**2 / (4 * mu_blood)  # Maximum velocity (centerline)
u_avg = u_max / 2       # Average velocity for Poiseuille flow
Re = rho_blood * u_avg * (2 * R_artery) / mu_blood  # Reynolds number

# Thermal parameters
T_inlet = 37.0          # Inlet blood temperature (°C)
T_wall = 35.0           # Arterial wall temperature (°C)
q_met = 500.0           # Metabolic heat generation (W/m³) in surrounding tissue

# Grid setup
nr = 40     # Radial points
nz = 80     # Axial points

r = np.linspace(0, R_artery, nr)
z = np.linspace(0, L_artery, nz)
dr = r[1] - r[0]
dz = z[1] - z[0]

R_grid, Z_grid = np.meshgrid(r, z)

print("Blood Flow Problem Setup:")
print(f"  Artery radius: {R_artery*1000:.1f} mm")
print(f"  Length: {L_artery*100:.1f} cm")
print(f"  Maximum velocity: {u_max*100:.2f} cm/s")
print(f"  Average velocity: {u_avg*100:.2f} cm/s")
print(f"  Reynolds number: {Re:.1f} (laminar flow)")
print(f"  Grid: {nr} × {nz} (r × z)")
print(f"  Thermal diffusivity: {alpha_blood:.3e} m²/s")

In [ ]:
# Calculate velocity field (analytical Poiseuille solution)
u_z = np.zeros((nz, nr))
for i in range(nr):
    u_z[:, i] = -dp_dz / (4 * mu_blood) * (R_artery**2 - r[i]**2)

# Verify flow rate
Q = 2 * np.pi * np.trapz(r * u_z[0, :], r)  # Volume flow rate (m³/s)
Q_ml_min = Q * 1e6 * 60  # Convert to mL/min

print("Velocity Field:")
print(f"  Centerline velocity: {u_z[0, 0]*100:.2f} cm/s")
print(f"  Volume flow rate: {Q_ml_min:.2f} mL/min")

# Initialize temperature field
T_blood = np.ones((nz, nr)) * T_inlet
T_blood[:, -1] = T_wall  # Wall boundary condition

# Solve energy equation using finite differences
# Marching in z-direction (convection dominated)
print("\nSolving energy equation...")

for j in range(1, nz):
    T_old = T_blood[j, :].copy()
    
    # Iterative solution at each axial location
    for iteration in range(100):
        T_new = T_blood[j, :].copy()
        
        # Interior points (excluding centerline and wall)
        for i in range(1, nr-1):
            r_i = r[i]
            r_plus = (r[i] + r[i+1]) / 2
            r_minus = (r[i] + r[i-1]) / 2
            
            # Radial conduction term
            d2T_dr2 = (r_plus * (T_new[i+1] - T_new[i]) - 
                      r_minus * (T_new[i] - T_new[i-1])) / (r_i * dr**2)
            
            # Axial convection term (upwind differencing)
            dT_dz = (T_new[i] - T_blood[j-1, i]) / dz
            
            # Energy balance
            T_new[i] = T_old[i] + (k_blood / (rho_blood * cp_blood * u_z[j, i] + 1e-10)) * (
                d2T_dr2 * dz + q_met * dz / (rho_blood * cp_blood)
            )
        
        # Centerline (r=0): symmetry condition dT/dr = 0
        T_new[0] = T_new[1]
        
        # Wall boundary condition
        T_new[-1] = T_wall
        
        # Check convergence
        if np.max(np.abs(T_new - T_blood[j, :])) < 1e-6:
            break
        
        T_blood[j, :] = T_new
    
    if j % 20 == 0:
        print(f"  z = {z[j]*100:.2f} cm: T_centerline = {T_blood[j, 0]:.3f}°C, "
              f"T_mean = {np.mean(T_blood[j, :]):.3f}°C")

print("\n✓ Solution complete!")
print(f"  Exit temperature (centerline): {T_blood[-1, 0]:.2f}°C")
print(f"  Exit temperature (mean): {np.mean(T_blood[-1, :]):.2f}°C")

In [ ]:
# Calculate bulk mean temperature
T_bulk = np.zeros(nz)
for j in range(nz):
    # Flow-weighted average
    numerator = 2 * np.pi * np.trapz(r * u_z[j, :] * T_blood[j, :], r)
    denominator = Q
    T_bulk[j] = numerator / denominator

# Calculate heat transfer coefficient
h = np.zeros(nz)
for j in range(nz):
    if T_bulk[j] - T_wall > 0.01:
        # Newton's law of cooling: q = h(T_bulk - T_wall)
        q_wall = -k_blood * (T_blood[j, -1] - T_blood[j, -2]) / dr
        h[j] = q_wall / (T_bulk[j] - T_wall)

# Calculate Nusselt number
D = 2 * R_artery
Nu = h * D / k_blood

print("Heat Transfer Analysis:")
print(f"  Average Nusselt number: {np.mean(Nu[Nu>0]):.2f}")
print(f"  Bulk temperature change: {T_bulk[-1] - T_bulk[0]:.3f}°C")

In [ ]:
# Visualization - Blood Flow in Artery
fig = plt.figure(figsize=(18, 14))

# 1. Temperature distribution (2D r-z plane)
ax1 = plt.subplot(3, 3, 1)
contourf1 = ax1.contourf(Z_grid*1000, R_grid*1000, T_blood, 
                         levels=30, cmap='RdYlBu_r')
contour1 = ax1.contour(Z_grid*1000, R_grid*1000, T_blood, 
                       levels=8, colors='black', alpha=0.3, linewidths=0.5)
ax1.clabel(contour1, inline=True, fontsize=8, fmt='%.1f°C')
plt.colorbar(contourf1, ax=ax1, label='Temperature (°C)')
ax1.set_xlabel('Axial Position z (mm)')
ax1.set_ylabel('Radius r (mm)')
ax1.set_title('Temperature Distribution in Blood Flow')
ax1.axhline(y=R_artery*1000, color='brown', linewidth=2, label='Arterial wall')
ax1.legend()

# 2. Velocity profile (parabolic)
ax2 = plt.subplot(3, 3, 2)
ax2.plot(u_z[0, :]*100, r*1000, 'b-', linewidth=2.5)
ax2.axvline(x=u_avg*100, color='r', linestyle='--', linewidth=2, 
            label=f'Average velocity ({u_avg*100:.2f} cm/s)')
ax2.set_xlabel('Velocity u_z (cm/s)')
ax2.set_ylabel('Radius (mm)')
ax2.set_title('Velocity Profile (Poiseuille Flow)')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.set_ylim([0, R_artery*1000])

# 3. Temperature profiles at different axial locations
ax3 = plt.subplot(3, 3, 3)
z_positions = [0, nz//4, nz//2, 3*nz//4, nz-1]
for j in z_positions:
    ax3.plot(r*1000, T_blood[j, :], linewidth=2, 
            label=f'z = {z[j]*1000:.1f} mm')
ax3.axhline(y=T_wall, color='brown', linestyle='--', alpha=0.5, label='Wall')
ax3.set_xlabel('Radius (mm)')
ax3.set_ylabel('Temperature (°C)')
ax3.set_title('Radial Temperature Profiles')
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=8)

# 4. Bulk temperature vs axial position
ax4 = plt.subplot(3, 3, 4)
ax4.plot(z*1000, T_bulk, 'r-', linewidth=2.5, label='Bulk temperature')
ax4.plot(z*1000, T_blood[:, 0], 'b--', linewidth=2, label='Centerline')
ax4.axhline(y=T_wall, color='brown', linestyle='--', linewidth=2, label='Wall')
ax4.axhline(y=T_inlet, color='g', linestyle=':', linewidth=1.5, alpha=0.5, label='Inlet')
ax4.set_xlabel('Axial Position (mm)')
ax4.set_ylabel('Temperature (°C)')
ax4.set_title('Axial Temperature Development')
ax4.grid(True, alpha=0.3)
ax4.legend()

# 5. Nusselt number distribution
ax5 = plt.subplot(3, 3, 5)
Nu_valid = Nu[Nu > 0]
z_valid = z[Nu > 0]
if len(Nu_valid) > 0:
    ax5.plot(z_valid*1000, Nu_valid, 'g-', linewidth=2.5)
    ax5.axhline(y=np.mean(Nu_valid), color='k', linestyle='--',
               label=f'Average Nu = {np.mean(Nu_valid):.2f}')
ax5.set_xlabel('Axial Position (mm)')
ax5.set_ylabel('Nusselt Number')
ax5.set_title('Local Nusselt Number')
ax5.grid(True, alpha=0.3)
ax5.legend()

# 6. 3D surface plot of temperature
ax6 = fig.add_subplot(3, 3, 6, projection='3d')
surf = ax6.plot_surface(Z_grid*1000, R_grid*1000, T_blood, 
                        cmap='RdYlBu_r', alpha=0.9)
ax6.set_xlabel('z (mm)')
ax6.set_ylabel('r (mm)')
ax6.set_zlabel('T (°C)')
ax6.set_title('3D Temperature Field')
ax6.view_init(elev=25, azim=45)

# 7. Full cross-section view (mirror for full circle)
ax7 = plt.subplot(3, 3, 7)
# Create full cross-section at exit
theta = np.linspace(0, 2*np.pi, 60)
R_theta, Theta = np.meshgrid(r, theta)
X = R_theta * np.cos(Theta) * 1000
Y = R_theta * np.sin(Theta) * 1000
T_exit_2d = np.tile(T_blood[-1, :], (len(theta), 1))
contourf7 = ax7.contourf(X, Y, T_exit_2d, levels=20, cmap='RdYlBu_r')
plt.colorbar(contourf7, ax=ax7, label='T (°C)')
ax7.set_xlabel('x (mm)')
ax7.set_ylabel('y (mm)')
ax7.set_title(f'Exit Cross-Section (z = {L_artery*1000:.0f} mm)')
ax7.set_aspect('equal')

# 8. Velocity magnitude with streamlines representation
ax8 = plt.subplot(3, 3, 8)
contourf8 = ax8.contourf(Z_grid*1000, R_grid*1000, u_z*100, 
                         levels=20, cmap='viridis')
plt.colorbar(contourf8, ax=ax8, label='Velocity (cm/s)')
# Add flow direction arrows
skip_r = 5
skip_z = 8
ax8.quiver(Z_grid[::skip_z, ::skip_r]*1000, R_grid[::skip_z, ::skip_r]*1000,
          np.ones_like(u_z[::skip_z, ::skip_r]), 
          np.zeros_like(u_z[::skip_z, ::skip_r]),
          alpha=0.6, color='white')
ax8.set_xlabel('Axial Position z (mm)')
ax8.set_ylabel('Radius r (mm)')
ax8.set_title('Velocity Field with Flow Direction')

# 9. Temperature difference from wall
ax9 = plt.subplot(3, 3, 9)
T_diff = T_blood - T_wall
contourf9 = ax9.contourf(Z_grid*1000, R_grid*1000, T_diff, 
                         levels=20, cmap='hot')
plt.colorbar(contourf9, ax=ax9, label='ΔT from wall (°C)')
ax9.set_xlabel('Axial Position z (mm)')
ax9.set_ylabel('Radius r (mm)')
ax9.set_title('Temperature Difference from Wall')

plt.tight_layout()
plt.savefig('../../figures/blood_flow_steady_cylindrical.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/blood_flow_steady_cylindrical.png")

### Biomedical Engineering Analysis

**Key Insights:**

1. **Thermal Entrance Length**: Temperature profile develops gradually from inlet
   - Fully developed thermal profile requires significant length
   - Important for drug delivery and hypothermia treatment design

2. **Radial Temperature Gradients**: 
   - Blood near wall is cooler than centerline
   - Parabolic velocity profile affects heat transfer
   - Faster flow at center carries heat further downstream

3. **Clinical Applications**:
   - **Hypothermia Treatment**: Cooling rates depend on flow velocity and vessel size
   - **Thermal Ablation**: Understanding temperature distribution critical for treatment planning
   - **Perfusion Measurement**: Temperature changes can indicate blood flow rates

4. **Nusselt Number Analysis**:
   - Entrance region shows higher heat transfer (developing boundary layer)
   - Approaches constant value in fully developed region
   - Typical Nu ≈ 3.66 for constant wall temperature in circular tube

**Physiological Relevance**:
- Blood flow in extremities (arms, legs) experiences cooling from environment
- Body regulates temperature through vasodilation/vasoconstriction
- Exercise increases flow rate, affecting heat transfer
- Important for understanding frostbite, thermal injuries

---
## Part 2: Transient Natural Convection in Cartesian Coordinates

### Application: Natural Convection in a Heated Bioreactor (Chemical Engineering)

Consider a 3-D rectangular bioreactor with:
- Bottom surface heated to maintain optimal growth temperature
- Cooler top surface (heat loss to environment)
- Vertical walls insulated
- Natural convection develops due to density differences
- Time-dependent startup from rest

### Engineering Significance
Natural convection is critical in:
- Bioreactor mixing and temperature uniformity
- Cell culture systems (shear stress from flow affects cells)
- Scale-up of batch processes
- Safety analysis (hot spots in reactors)
- Energy efficiency (passive mixing vs. mechanical agitation)

### Governing Equations (3-D Cartesian: x, y, z)

**Continuity:**
$$\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} + \frac{\partial w}{\partial z} = 0$$

**Momentum (with Boussinesq approximation):**
$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} + w\frac{\partial u}{\partial z} = -\frac{1}{\rho}\frac{\partial p}{\partial x} + \nu\nabla^2 u$$

$$\frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} + w\frac{\partial v}{\partial z} = -\frac{1}{\rho}\frac{\partial p}{\partial y} + \nu\nabla^2 v$$

$$\frac{\partial w}{\partial t} + u\frac{\partial w}{\partial x} + v\frac{\partial w}{\partial y} + w\frac{\partial w}{\partial z} = -\frac{1}{\rho}\frac{\partial p}{\partial z} + \nu\nabla^2 w - g\beta(T - T_0)$$

**Energy:**
$$\frac{\partial T}{\partial t} + u\frac{\partial T}{\partial x} + v\frac{\partial T}{\partial y} + w\frac{\partial T}{\partial z} = \alpha\nabla^2 T$$

where $\beta$ is the thermal expansion coefficient.

### Dimensionless Parameters

**Rayleigh Number**: $Ra = \frac{g\beta\Delta T L^3}{\nu\alpha}$ (ratio of buoyancy to viscous forces)

**Prandtl Number**: $Pr = \frac{\nu}{\alpha}$ (ratio of momentum to thermal diffusivity)

### Numerical Approach
For this simplified model, we'll solve:
1. 2-D slice (x-z plane) assuming uniformity in y-direction
2. Use stream function-vorticity formulation to satisfy continuity
3. Explicit time-stepping for transient behavior

In [ ]:
# Problem parameters - Bioreactor Natural Convection
# Geometric parameters (2D slice for computational efficiency)
L_reactor = 0.1      # Width (m) - 10 cm
H_reactor = 0.15     # Height (m) - 15 cm

# Fluid properties (water-based culture medium)
rho_medium = 1000.0     # Density (kg/m³)
mu_medium = 0.001       # Dynamic viscosity (Pa·s)
nu_medium = mu_medium / rho_medium  # Kinematic viscosity (m²/s)
k_medium = 0.6          # Thermal conductivity (W/m·K)
cp_medium = 4180.0      # Specific heat (J/kg·K)
alpha_medium = k_medium / (rho_medium * cp_medium)
beta_medium = 2.1e-4    # Thermal expansion coefficient (1/K) for water

# Temperature conditions
T_hot = 40.0        # Bottom heated surface (°C) - optimal growth temp
T_cold = 30.0       # Top surface (°C) - cooler due to evaporation
T_initial = 30.0    # Initial uniform temperature (°C)
Delta_T = T_hot - T_cold

# Gravity
g = 9.81  # m/s²

# Calculate dimensionless numbers
Ra = g * beta_medium * Delta_T * H_reactor**3 / (nu_medium * alpha_medium)
Pr = nu_medium / alpha_medium

# Grid setup
nx = 60
nz = 80
dx = L_reactor / (nx - 1)
dz = H_reactor / (nz - 1)

x = np.linspace(0, L_reactor, nx)
z = np.linspace(0, H_reactor, nz)
X, Z = np.meshgrid(x, z)

# Time stepping
dt_max = 0.2 * min(dx**2, dz**2) / (4 * max(nu_medium, alpha_medium))
dt = 0.5 * dt_max
total_time = 300.0  # 5 minutes
nt = int(total_time / dt)

print("Bioreactor Natural Convection Setup:")
print(f"  Dimensions: {L_reactor*100:.1f} cm × {H_reactor*100:.1f} cm")
print(f"  Temperature difference: {Delta_T:.1f}°C")
print(f"  Rayleigh number: {Ra:.2e}")
print(f"  Prandtl number: {Pr:.2f}")
if Ra < 1708:
    print(f"  Flow regime: Conduction dominated (Ra < 1708)")
elif Ra < 1e6:
    print(f"  Flow regime: Laminar natural convection")
else:
    print(f"  Flow regime: Turbulent natural convection")
print(f"  Grid: {nx} × {nz}")
print(f"  Time step: {dt:.4f} s")
print(f"  Total time steps: {nt}")

In [ ]:
# Initialize fields
T = np.ones((nz, nx)) * T_initial  # Temperature
u = np.zeros((nz, nx))  # x-velocity
w = np.zeros((nz, nx))  # z-velocity (vertical)
p = np.zeros((nz, nx))  # Pressure (relative)

# Storage for visualization
snapshot_times_reactor = [0, 30, 60, 120, 180, 300]
snapshots_T = []
snapshots_u = []
snapshots_w = []
time_history_reactor = []
Nu_history = []

print("\nSolving transient natural convection...")
print("Progress: ", end='')

for n in range(nt):
    current_time = n * dt
    
    # Save old values
    T_old = T.copy()
    u_old = u.copy()
    w_old = w.copy()
    
    # Update temperature (energy equation) - explicit
    for i in range(1, nz-1):
        for j in range(1, nx-1):
            # Convection terms
            conv_T = (u_old[i, j] * (T_old[i, j+1] - T_old[i, j-1]) / (2*dx) +
                     w_old[i, j] * (T_old[i+1, j] - T_old[i-1, j]) / (2*dz))
            
            # Diffusion terms
            diff_T = alpha_medium * (
                (T_old[i, j+1] - 2*T_old[i, j] + T_old[i, j-1]) / dx**2 +
                (T_old[i+1, j] - 2*T_old[i, j] + T_old[i-1, j]) / dz**2
            )
            
            T[i, j] = T_old[i, j] + dt * (-conv_T + diff_T)
    
    # Apply temperature boundary conditions
    T[0, :] = T_hot      # Bottom (heated)
    T[-1, :] = T_cold    # Top (cooled)
    T[:, 0] = T[:, 1]    # Left wall (insulated - zero gradient)
    T[:, -1] = T[:, -2]  # Right wall (insulated - zero gradient)
    
    # Update velocities (simplified momentum equations)
    for i in range(1, nz-1):
        for j in range(1, nx-1):
            # x-momentum (horizontal)
            conv_u = (u_old[i, j] * (u_old[i, j+1] - u_old[i, j-1]) / (2*dx) +
                     w_old[i, j] * (u_old[i+1, j] - u_old[i-1, j]) / (2*dz))
            
            diff_u = nu_medium * (
                (u_old[i, j+1] - 2*u_old[i, j] + u_old[i, j-1]) / dx**2 +
                (u_old[i+1, j] - 2*u_old[i, j] + u_old[i-1, j]) / dz**2
            )
            
            # Pressure gradient (simplified)
            dp_dx = (p[i, j+1] - p[i, j-1]) / (2*dx)
            
            u[i, j] = u_old[i, j] + dt * (-conv_u + diff_u - dp_dx/rho_medium)
            
            # z-momentum (vertical) - includes buoyancy
            conv_w = (u_old[i, j] * (w_old[i, j+1] - w_old[i, j-1]) / (2*dx) +
                     w_old[i, j] * (w_old[i+1, j] - w_old[i-1, j]) / (2*dz))
            
            diff_w = nu_medium * (
                (w_old[i, j+1] - 2*w_old[i, j] + w_old[i, j-1]) / dx**2 +
                (w_old[i+1, j] - 2*w_old[i, j] + w_old[i-1, j]) / dz**2
            )
            
            # Pressure gradient
            dp_dz = (p[i+1, j] - p[i-1, j]) / (2*dz)
            
            # Buoyancy force (Boussinesq approximation)
            buoyancy = g * beta_medium * (T[i, j] - T_cold)
            
            w[i, j] = w_old[i, j] + dt * (-conv_w + diff_w - dp_dz/rho_medium + buoyancy)
    
    # Apply velocity boundary conditions (no-slip at walls)
    u[:, 0] = 0
    u[:, -1] = 0
    u[0, :] = 0
    u[-1, :] = 0
    
    w[:, 0] = 0
    w[:, -1] = 0
    w[0, :] = 0
    w[-1, :] = 0
    
    # Update pressure (from continuity - Poisson equation, simplified)
    for iteration in range(20):  # Pressure correction iterations
        p_old = p.copy()
        for i in range(1, nz-1):
            for j in range(1, nx-1):
                # Divergence of velocity
                div_u = (u[i, j+1] - u[i, j-1]) / (2*dx) + (w[i+1, j] - w[i-1, j]) / (2*dz)
                
                # Pressure Poisson equation
                p[i, j] = 0.25 * (p_old[i+1, j] + p_old[i-1, j] + 
                                 p_old[i, j+1] + p_old[i, j-1]) - \
                         0.25 * rho_medium * div_u * (dx**2 + dz**2) / 2
        
        # Pressure boundary conditions (zero gradient at walls)
        p[0, :] = p[1, :]
        p[-1, :] = p[-2, :]
        p[:, 0] = p[:, 1]
        p[:, -1] = p[:, -2]
    
    # Calculate Nusselt number at bottom wall
    if current_time > 0:
        q_bottom = -k_medium * (T[1, :] - T[0, :]) / dz
        h_bottom = q_bottom / (T_hot - T_cold + 1e-10)
        Nu_bottom = np.mean(h_bottom) * H_reactor / k_medium
    else:
        Nu_bottom = 0
    
    # Save snapshots
    if current_time in snapshot_times_reactor or \
       (len(snapshots_T) < len(snapshot_times_reactor) and 
        current_time >= snapshot_times_reactor[len(snapshots_T)]):
        if len(snapshots_T) < len(snapshot_times_reactor):
            snapshots_T.append(T.copy())
            snapshots_u.append(u.copy())
            snapshots_w.append(w.copy())
            time_history_reactor.append(current_time)
            Nu_history.append(Nu_bottom)
            max_vel = np.sqrt(np.max(u**2 + w**2))
            print(f"\n  t = {current_time:.0f} s: T_max = {T.max():.2f}°C, "
                  f"T_min = {T.min():.2f}°C, max_vel = {max_vel*1000:.2f} mm/s, Nu = {Nu_bottom:.2f}", 
                  end='')
    
    # Progress indicator
    if n % (nt // 20) == 0:
        print('.', end='', flush=True)

print("\n\n✓ Simulation complete!")
print(f"  Final temperature range: {T.min():.2f} - {T.max():.2f}°C")
print(f"  Final maximum velocity: {np.sqrt(np.max(u**2 + w**2))*1000:.2f} mm/s")
print(f"  Final Nusselt number: {Nu_bottom:.2f}")

In [ ]:
# Visualization - Natural Convection in Bioreactor
fig = plt.figure(figsize=(20, 16))

# Plot snapshots of temperature and velocity
for idx, (T_snap, u_snap, w_snap, t) in enumerate(
    zip(snapshots_T, snapshots_u, snapshots_w, time_history_reactor)):
    
    ax = plt.subplot(4, 4, idx + 1)
    
    # Temperature contours
    contourf = ax.contourf(X*100, Z*100, T_snap, 
                           levels=np.linspace(T_cold, T_hot, 20),
                           cmap='RdYlBu_r', vmin=T_cold, vmax=T_hot)
    plt.colorbar(contourf, ax=ax, label='T (°C)')
    
    # Velocity vectors
    skip = 4
    vel_magnitude = np.sqrt(u_snap**2 + w_snap**2)
    quiver = ax.quiver(X[::skip, ::skip]*100, Z[::skip, ::skip]*100,
                      u_snap[::skip, ::skip], w_snap[::skip, ::skip],
                      vel_magnitude[::skip, ::skip]*1000,
                      cmap='cool', alpha=0.7, scale=0.01, width=0.003)
    
    ax.set_xlabel('Width (cm)')
    ax.set_ylabel('Height (cm)')
    ax.set_title(f't = {t:.0f} s ({t/60:.1f} min)')
    ax.set_aspect('equal')
    
    # Mark hot and cold walls
    ax.text(L_reactor*50, 0.2, 'HOT', ha='center', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='red', alpha=0.7))
    ax.text(L_reactor*50, H_reactor*100-0.3, 'COLD', ha='center', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='blue', alpha=0.7))

# Nusselt number evolution
ax7 = plt.subplot(4, 4, 7)
ax7.plot(np.array(time_history_reactor), Nu_history, 'r-o', linewidth=2.5, markersize=8)
ax7.set_xlabel('Time (s)')
ax7.set_ylabel('Nusselt Number')
ax7.set_title('Heat Transfer Intensity vs Time')
ax7.grid(True, alpha=0.3)

# Temperature at different heights vs time
ax8 = plt.subplot(4, 4, 8)
heights = [nz//4, nz//2, 3*nz//4]
height_labels = ['1/4 height', '1/2 height', '3/4 height']
for h, label in zip(heights, height_labels):
    temps = [T_snap[h, nx//2] for T_snap in snapshots_T]
    ax8.plot(time_history_reactor, temps, '-o', linewidth=2, label=label)
ax8.axhline(y=T_hot, color='r', linestyle='--', alpha=0.5, label='Hot wall')
ax8.axhline(y=T_cold, color='b', linestyle='--', alpha=0.5, label='Cold wall')
ax8.set_xlabel('Time (s)')
ax8.set_ylabel('Temperature (°C)')
ax8.set_title('Temperature History at Centerline')
ax8.grid(True, alpha=0.3)
ax8.legend(fontsize=8)

# Final velocity magnitude
ax9 = plt.subplot(4, 4, 9)
vel_mag_final = np.sqrt(snapshots_u[-1]**2 + snapshots_w[-1]**2) * 1000
contourf9 = ax9.contourf(X*100, Z*100, vel_mag_final, levels=20, cmap='viridis')
plt.colorbar(contourf9, ax=ax9, label='Velocity (mm/s)')
# Streamlines
ax9.streamplot(X*100, Z*100, snapshots_u[-1], snapshots_w[-1], 
              color='white', density=1.5, linewidth=1, arrowsize=1.5)
ax9.set_xlabel('Width (cm)')
ax9.set_ylabel('Height (cm)')
ax9.set_title('Final Velocity Field with Streamlines')
ax9.set_aspect('equal')

# Vertical temperature profile at final time
ax10 = plt.subplot(4, 4, 10)
ax10.plot(snapshots_T[-1][:, nx//4], z*100, linewidth=2, label='x = L/4')
ax10.plot(snapshots_T[-1][:, nx//2], z*100, linewidth=2, label='x = L/2')
ax10.plot(snapshots_T[-1][:, 3*nx//4], z*100, linewidth=2, label='x = 3L/4')
ax10.set_xlabel('Temperature (°C)')
ax10.set_ylabel('Height (cm)')
ax10.set_title('Final Vertical Temperature Profiles')
ax10.grid(True, alpha=0.3)
ax10.legend()

# Maximum velocity vs time
ax11 = plt.subplot(4, 4, 11)
max_vels = [np.sqrt(np.max(u**2 + w**2))*1000 
           for u, w in zip(snapshots_u, snapshots_w)]
ax11.plot(time_history_reactor, max_vels, 'b-o', linewidth=2.5, markersize=8)
ax11.set_xlabel('Time (s)')
ax11.set_ylabel('Maximum Velocity (mm/s)')
ax11.set_title('Flow Development')
ax11.grid(True, alpha=0.3)

# 3D temperature surface at final time
ax12 = fig.add_subplot(4, 4, 12, projection='3d')
surf = ax12.plot_surface(X*100, Z*100, snapshots_T[-1], 
                        cmap='RdYlBu_r', alpha=0.9)
ax12.set_xlabel('Width (cm)')
ax12.set_ylabel('Height (cm)')
ax12.set_zlabel('Temperature (°C)')
ax12.set_title('3D Temperature Field (final)')
ax12.view_init(elev=25, azim=45)

plt.tight_layout()
plt.savefig('../../figures/bioreactor_natural_convection_transient_cartesian.png', 
           dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/bioreactor_natural_convection_transient_cartesian.png")

### Chemical Engineering Analysis - Bioreactor

**Key Findings:**

1. **Flow Pattern Development**:
   - Initially stagnant fluid begins to move as temperature gradients develop
   - Hot fluid rises near bottom (buoyancy force)
   - Cold fluid descends near top
   - Recirculation cells form - natural mixing

2. **Temperature Distribution**:
   - Strong vertical stratification initially (pure conduction)
   - Convection develops and redistributes heat
   - Boundary layers form near heated/cooled surfaces
   - Final state shows more uniform temperature (better mixing)

3. **Nusselt Number Evolution**:
   - Nu = 1 initially (pure conduction)
   - Increases as convection develops
   - Higher Nu means better heat transfer
   - Steady-state Nu depends on Rayleigh number

4. **Velocity Magnitude**:
   - Starts at zero (no motion)
   - Grows as buoyancy forces overcome inertia
   - Typical velocities: mm/s (gentle mixing)
   - Low shear stress - suitable for cell cultures

**Design Implications:**

1. **Mixing Performance**:
   - Natural convection provides passive mixing
   - No mechanical agitation needed (reduces cost, complexity)
   - Gentle flow suitable for shear-sensitive cells
   - May be insufficient for fast reactions or high-density cultures

2. **Temperature Control**:
   - Heating from bottom creates instability (promotes mixing)
   - Cooling from top helps maintain stratification
   - Hot spots can affect cell viability
   - Need to consider transient startup behavior

3. **Scale-Up Considerations**:
   - Rayleigh number scales with L³ (vessel size)
   - Larger vessels → stronger natural convection
   - Flow patterns may change (laminar → turbulent)
   - Heat transfer characteristics differ at large scale

4. **Process Optimization**:
   - Balance temperature uniformity vs. shear stress
   - Consider startup time to reach steady state
   - Monitor local conditions (not just bulk average)
   - Validate with CFD for complex geometries

5. **Safety Considerations**:
   - Avoid runaway heating (exothermic reactions)
   - Ensure adequate cooling capacity
   - Consider heat generation from cells/reactions
   - Emergency cooling strategies

---
## Summary: 3-D Fluid Dynamics with Heat Transfer

### Coordinate System Comparison

| Aspect | Cylindrical | Cartesian |
|--------|------------|----------|
| **Best for** | Pipes, vessels, arteries | Rectangular reactors, channels |
| **Complexity** | Weighted derivatives, r=0 special case | Uniform derivatives |
| **Common applications** | Blood vessels, pipe flow, can sterilization | Bioreactors, cavities, plate flow |
| **Velocity profile** | Parabolic (Poiseuille) for laminar | Varies with geometry |

### Steady vs. Transient Analysis

**Steady-State (Blood Flow Example)**:
- Fully developed flow and thermal profiles
- Faster computation (no time stepping)
- Good for design calculations (heat exchangers, etc.)
- Analytical solutions possible for simple geometries
- Focus: Entrance effects, Nusselt numbers, pressure drop

**Transient (Bioreactor Example)**:
- Flow development from rest
- Time-dependent boundary conditions
- Startup/shutdown analysis
- Natural convection development
- Focus: Response time, stability, control strategies

### Coupling of Momentum and Energy

1. **Forced Convection** (Blood flow):
   - Velocity field independent of temperature
   - Solve momentum first, then energy
   - One-way coupling

2. **Natural Convection** (Bioreactor):
   - Velocity depends on temperature (buoyancy)
   - Temperature depends on velocity (convection)
   - Two-way coupling - solve simultaneously

### Dimensionless Numbers

**Reynolds Number** ($Re$): Inertia vs. viscous forces
- $Re < 2300$: Laminar flow
- $Re > 4000$: Turbulent flow

**Prandtl Number** ($Pr$): Momentum vs. thermal diffusivity
- $Pr < 1$: Liquid metals
- $Pr \approx 1$: Gases
- $Pr > 1$: Oils, water

**Rayleigh Number** ($Ra$): Buoyancy vs. viscous forces
- $Ra < 10^3$: Conduction dominated
- $10^3 < Ra < 10^6$: Laminar convection
- $Ra > 10^6$: Turbulent convection

**Nusselt Number** ($Nu$): Convective vs. conductive heat transfer
- $Nu = 1$: Pure conduction
- $Nu > 1$: Convection enhances heat transfer

**Peclet Number** ($Pe = Re \cdot Pr$): Convection vs. conduction
- $Pe >> 1$: Convection dominated
- $Pe << 1$: Conduction dominated

### Applications in Chemical and Biomolecular Engineering

**Biomolecular Engineering**:
1. Blood flow and oxygen/nutrient transport
2. Bioreactor design (cell culture, fermentation)
3. Drug delivery systems
4. Artificial organs (dialysis, oxygenators)
5. Thermal therapy (hyperthermia, cryotherapy)

**Chemical Engineering**:
1. Tubular reactors (plug flow, laminar flow reactors)
2. Heat exchangers (shell-and-tube, plate)
3. Distillation columns (tray efficiency)
4. Crystallization (supersaturation, nucleation)
5. Polymerization reactors (temperature control)

### Computational Considerations

**Grid Resolution**:
- Boundary layers require fine meshes
- Grid independence studies essential
- Adaptive meshing for efficiency

**Time Stepping** (transient):
- Stability criteria (CFL condition)
- Explicit vs. implicit methods
- Adaptive time stepping

**Convergence**:
- Iterative methods for coupled equations
- Under-relaxation for stability
- Residual monitoring

**Validation**:
- Comparison with analytical solutions
- Experimental data
- Benchmark problems

### Advanced Topics

1. **Turbulence Modeling**: k-ε, k-ω, LES, DNS
2. **Multiphase Flow**: Gas-liquid, solid-liquid
3. **Non-Newtonian Fluids**: Blood, polymers, slurries
4. **Reactive Flows**: Combustion, polymerization
5. **Porous Media**: Filters, packed beds, tissue

### Software Tools

**Commercial CFD**:
- ANSYS Fluent
- COMSOL Multiphysics
- STAR-CCM+

**Open Source**:
- OpenFOAM
- SU2
- FEniCS

**Python Libraries**:
- FiPy (finite volume)
- SfePy (finite element)
- PyFR (high-order methods)

---
## Exercises

### Exercise 1: Effect of Flow Rate on Blood Temperature
Modify the blood flow example to:
- Vary the pressure gradient (change flow rate)
- Calculate exit temperature for different Reynolds numbers
- Plot Nu vs. Re and compare with correlations
- Discuss implications for blood flow regulation

### Exercise 2: Pulsatile Blood Flow
Extend the steady blood flow to transient:
- Add time-varying pressure gradient (sinusoidal)
- Simulate cardiac cycle (systole/diastole)
- Analyze unsteady heat transfer
- Compare with steady-flow approximation

### Exercise 3: Bioreactor with Heat Generation
Modify the bioreactor example:
- Add volumetric heat generation (exothermic reaction or cell metabolism)
- Study effect on flow patterns and temperature
- Determine maximum safe heat generation rate
- Design cooling strategy

### Exercise 4: Different Rayleigh Numbers
For the natural convection problem:
- Vary ΔT to change Rayleigh number
- Compare flow patterns for Ra = 10³, 10⁴, 10⁵, 10⁶
- Plot Nu vs. Ra (log-log)
- Compare with empirical correlations

### Exercise 5: Graetz Problem
Classical heat transfer problem:
- Cylindrical pipe with fully developed velocity
- Step change in wall temperature at z = 0
- Solve for thermal entrance length
- Compare with analytical solution

### Exercise 6: Lid-Driven Cavity with Heating
Combine forced and natural convection:
- Top wall moving (creates forced convection)
- Bottom wall heated (creates natural convection)
- Study Richardson number (Ri = Gr/Re²)
- Identify forced vs. natural convection dominated regimes

### Exercise 7: Blood Vessel with Non-Newtonian Rheology
More realistic blood model:
- Implement power-law or Casson model for viscosity
- Compare velocity profiles with Newtonian case
- Analyze effect on heat transfer
- Discuss physiological relevance

### Exercise 8: 3-D Extension
Add third dimension:
- Solve full 3-D problem (x, y, z)
- Consider azimuthal variations (θ in cylindrical)
- Analyze computational cost
- Determine when 2-D approximation is valid

---
## References and Further Reading

### Textbooks
1. **Incropera, F.P., DeWitt, D.P.** *Fundamentals of Heat and Mass Transfer* - Classic heat transfer text
2. **Bird, R.B., Stewart, W.E., Lightfoot, E.N.** *Transport Phenomena* - Comprehensive transport theory
3. **Bejan, A.** *Convection Heat Transfer* - Detailed convection analysis
4. **White, F.M.** *Viscous Fluid Flow* - Advanced fluid mechanics
5. **Geankoplis, C.J.** *Transport Processes and Separation Process Principles* - ChE applications

### Biomedical Applications
1. **Cooney, D.O.** *Biomedical Engineering Principles* - Medical applications
2. **Fournier, R.L.** *Basic Transport Phenomena in Biomedical Engineering* - Bio-specific
3. **Waite, L., Fine, J.** *Applied Biofluid Mechanics* - Blood flow and physiological fluids

### Computational Methods
1. **Patankar, S.V.** *Numerical Heat Transfer and Fluid Flow* - Finite volume methods
2. **Ferziger, J.H., Perić, M.** *Computational Methods for Fluid Dynamics* - CFD fundamentals
3. **Anderson, J.D.** *Computational Fluid Dynamics* - Practical CFD

### Research Papers (Selected)
1. Graetz (1883) - Original Graetz problem solution
2. Womersley (1955) - Pulsatile blood flow theory
3. De Vahl Davis (1983) - Natural convection benchmark
4. Ghia et al. (1982) - Lid-driven cavity benchmark

### Online Resources
1. **CFD Online** - Forums and resources
2. **LearnChemE** - Educational videos and screencasts
3. **COMSOL Blog** - Application examples
4. **OpenFOAM Documentation** - Open-source CFD

---

**End of Notebook**

*ChBE 3300: Multidimensional Fluids and Heat Transport*